In [1]:
import pandas as pd

In [2]:
events = pd.read_csv("../data/events.csv")

In [3]:
events.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [4]:
events.shape

(2756101, 5)

In [5]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          str    
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), str(1)
memory usage: 105.1 MB


In [6]:
events["event"].value_counts()

event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

In [7]:
events["visitorid"].nunique()

1407580

In [8]:
events["itemid"].nunique()

235061

In [9]:
events.groupby("visitorid").size().describe()

count    1.407580e+06
mean     1.958042e+00
std      1.258049e+01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      7.757000e+03
dtype: float64

In [10]:
events.groupby("itemid").size().describe()

count    235061.000000
mean         11.725046
std          37.032101
min           1.000000
25%           1.000000
50%           3.000000
75%           9.000000
max        3412.000000
dtype: float64

In [11]:
events.isna().sum()

timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

In [12]:
events.groupby("event")["transactionid"].count()

event
addtocart          0
transaction    22457
view               0
Name: transactionid, dtype: int64

In [13]:
events["itemid"].value_counts().head(10)

itemid
187946    3412
461686    2978
5411      2334
370653    1854
219512    1800
257040    1647
298009    1642
96924     1633
309778    1628
384302    1608
Name: count, dtype: int64

In [14]:
transactions = events[events["event"] == "transaction"]

In [15]:
transactions["itemid"].value_counts().head(10)

itemid
461686    133
119736     97
213834     92
7943       46
312728     46
445351     45
48030      41
420960     38
248455     38
17478      37
Name: count, dtype: int64

In [16]:
events[events["itemid"] == 461686]["event"].value_counts()

event
view           2539
addtocart       306
transaction     133
Name: count, dtype: int64

In [17]:
pd.to_datetime(events["timestamp"], unit="ms").min()

Timestamp('2015-05-03 03:00:04.384000')

In [18]:
pd.to_datetime(events["timestamp"], unit="ms").max()

Timestamp('2015-09-18 02:59:47.788000')

In [19]:
events.duplicated().sum()

np.int64(460)

In [20]:
events_clean = events.drop_duplicates()

In [21]:
events_clean.shape

(2755641, 5)

In [22]:
events_clean.duplicated().sum()

np.int64(0)

In [23]:
events_clean["datetime"] = pd.to_datetime(events_clean["timestamp"], unit="ms")

In [24]:
events_clean["datetime"].dt.to_period("M").value_counts().sort_index()

datetime
2015-05    590584
2015-06    610313
2015-07    697849
2015-08    553237
2015-09    303658
Freq: M, Name: count, dtype: int64

In [25]:
train_temp = events_clean[events_clean["datetime"] < "2015-09-01"]
test_temp = events_clean[events_clean["datetime"] >= "2015-09-01"]

In [26]:
train_users = set(train_temp["visitorid"].unique())
test_users = set(test_temp["visitorid"].unique())

len(test_users), len(test_users & train_users)

(173728, 17004)

In [27]:
returning_users = test_users & train_users

In [28]:
returning_user_history = (
    train_temp[train_temp["visitorid"].isin(returning_users)]
    .groupby("visitorid")
    .size()
)

In [29]:
returning_user_history.describe()

count    17004.000000
mean         7.048694
std         72.268954
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max       6683.000000
dtype: float64

In [30]:
for threshold in [1, 2, 3, 5, 10]:
    count = (returning_user_history >= threshold).sum()
    print(threshold, count)

1 17004
2 9154
3 6172
5 3629
10 1656


In [31]:
eligible_users = returning_user_history[
    returning_user_history >= 3
].index

In [32]:
len(eligible_users)

6172

In [33]:
test_eligible = test_temp[
    test_temp["visitorid"].isin(eligible_users)
]

In [34]:
strong_test = test_eligible[
    test_eligible["event"].isin(["addtocart", "transaction"])
]

In [35]:
strong_test["visitorid"].nunique()

324

In [36]:
test_eligible["visitorid"].nunique()

6172

In [37]:
train_transactions = train_temp[
    train_temp["event"] == "transaction"
]

In [38]:
popular_items = (
    train_transactions["itemid"]
    .value_counts()
    .index
    .tolist()
)

In [39]:
popular_items[:10]

[461686, 119736, 213834, 445351, 48030, 7943, 420960, 17478, 248455, 312728]

In [40]:
user_relevant_items = (
    test_eligible
    .groupby("visitorid")["itemid"]
    .apply(set)
)

In [41]:
user_relevant_items.head()

visitorid
155     {442601, 143373, 224623, 452082, 368372, 38997...
484                        {57610, 452803, 129046, 14606}
1764                                             {224481}
2004                                       {370368, 5661}
2043                                              {44871}
Name: itemid, dtype: object

In [42]:
first_user = user_relevant_items.index[0]

In [43]:
relevant = user_relevant_items.loc[first_user]
recommendations = set(popular_items[:10])

hits = recommendations & relevant

In [44]:
first_user, relevant, recommendations, hits

(np.int64(155),
 {41882, 143373, 181405, 224623, 368372, 389974, 442601, 452082},
 {7943, 17478, 48030, 119736, 213834, 248455, 312728, 420960, 445351, 461686},
 set())

In [45]:
def precision_at_k(relevant_items, recommended_items, k=10):
    recommended_k = set(recommended_items[:k])
    hits = recommended_k & relevant_items
    return len(hits) / k

In [46]:
def recall_at_k(relevant_items, recommended_items, k=10):
    recommended_k = set(recommended_items[:k])
    hits = recommended_k & relevant_items
    return len(hits) / len(relevant_items)

In [47]:
precision_scores = user_relevant_items.apply(
    lambda relevant: precision_at_k(relevant, popular_items, k=10)
)

recall_scores = user_relevant_items.apply(
    lambda relevant: recall_at_k(relevant, popular_items, k=10)
)

In [48]:
precision_scores.mean(), recall_scores.mean()

(np.float64(0.0015878159429682439), np.float64(0.0051663209591425825))

In [49]:
(precision_scores > 0).sum()

np.int64(79)

In [50]:
(precision_scores > 0).mean()

np.float64(0.012799740764744005)